# deployment_historical_risk_patch

Notebook hasil konversi dari `deployment_historical_risk_patch.py`. Kode dipisah ke beberapa cell berdasarkan blok top-level agar lebih mudah dibaca, dijalankan, dan di-screenshot.


## Imports


In [ ]:
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path


## Konstanta dan Setup Awal


In [ ]:
PROJECT_ROOT = Path(__file__).resolve().parents[1]


## Entry Point


In [ ]:
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## Code Block


In [ ]:
from backend import settings


## Code Block


In [ ]:
from backend.main import create_app


## Function: `build_parser`


In [ ]:
def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=(
            "Ringkasan fase Deployment historical risk patch. "
            "Script ini fokus pada wiring model ke backend FastAPI, runtime aktif, dan endpoint utama."
        )
    )
    parser.add_argument(
        "--show-routes",
        action="store_true",
        help="Tampilkan daftar route API dan halaman utama.",
    )
    parser.add_argument("--json", action="store_true")
    return parser


## Function: `collect_routes`


In [ ]:
def collect_routes(app) -> list[dict]:
    routes: list[dict] = []
    for route in app.routes:
        path = getattr(route, "path", None)
        methods = sorted(method for method in (getattr(route, "methods", set()) or set()) if method != "HEAD")
        name = getattr(route, "name", None)
        if not path:
            continue
        routes.append(
            {
                "path": path,
                "methods": methods,
                "name": name,
            }
        )
    return routes


## Function: `build_summary`


In [ ]:
def build_summary(args: argparse.Namespace) -> dict:
    app = create_app()
    model_spec = app.state.model_spec
    inference_service = app.state.inference_service
    routes = collect_routes(app)

    summary = {
        "phase": "deployment",
        "framework": "FastAPI",
        "server_entrypoint": "backend.main:app",
        "server_run_command": "python -m uvicorn backend.main:app --reload",
        "default_url": "http://127.0.0.1:8000",
        "active_model_profile": model_spec.profile_name,
        "active_model_display_name": model_spec.display_name,
        "preprocess_mode": model_spec.preprocess_mode,
        "inference_mode": model_spec.inference_mode,
        "time_steps": model_spec.time_steps,
        "grid_size": model_spec.grid_size,
        "channels": model_spec.channels,
        "patch_size": model_spec.patch_size,
        "patch_stride": model_spec.patch_stride,
        "patch_batch_size": model_spec.patch_batch_size,
        "input_dilation_kernel": model_spec.input_dilation_kernel,
        "recommended_threshold": model_spec.recommended_threshold,
        "prediction_engine": inference_service.backend,
        "model_backend": inference_service.backend,
        "model_path": inference_service.model_path,
        "runtime_model_loaded": inference_service.model is not None,
        "storage_dirs": {
            "inputs": str(settings.INPUTS_DIR),
            "outputs": str(settings.OUTPUTS_DIR),
            "templates": str(settings.TEMPLATES_DIR),
            "static": str(settings.STATIC_DIR),
        },
        "model_candidates": inference_service.list_model_candidates(),
        "key_routes": routes,
    }
    return summary


## Function: `print_human_summary`


In [ ]:
def print_human_summary(summary: dict, show_routes: bool) -> None:
    print("[deployment] Framework:", summary["framework"])
    print("[deployment] Entrypoint:", summary["server_entrypoint"])
    print("[deployment] Command run:", summary["server_run_command"])
    print("[deployment] URL default:", summary["default_url"])
    print("[deployment] Model profile aktif:", summary["active_model_profile"])
    print("[deployment] Nama model:", summary["active_model_display_name"])
    print("[deployment] Preprocess mode:", summary["preprocess_mode"])
    print("[deployment] Inference mode:", summary["inference_mode"])
    print("[deployment] Input:", f"{summary['time_steps']} frame | {summary['grid_size']}x{summary['grid_size']} | {summary['channels']} channel")
    print("[deployment] Patch config:", f"size={summary['patch_size']} stride={summary['patch_stride']} batch={summary['patch_batch_size']}")
    print("[deployment] Threshold rekomendasi:", summary["recommended_threshold"])
    print("[deployment] Prediction engine:", summary["prediction_engine"])
    print("[deployment] Model path:", summary["model_path"] or "-")
    print("[deployment] Runtime model loaded:", summary["runtime_model_loaded"])
    print("[deployment] Direktori storage:")
    for key, value in summary["storage_dirs"].items():
        print(f"- {key}: {value}")
    print("[deployment] Jumlah model candidate:", len(summary["model_candidates"]))

    if show_routes:
        print("[deployment] Routes:")
        for route in summary["key_routes"]:
            methods = ",".join(route["methods"]) if route["methods"] else "-"
            print(f"- {methods:12s} {route['path']} | name={route['name']}")


## Function: `main`


In [ ]:
def main() -> None:
    parser = build_parser()
    args = parser.parse_args()
    summary = build_summary(args)
    print_human_summary(summary, show_routes=args.show_routes)
    if args.json:
        print()
        print(json.dumps(summary, indent=2, ensure_ascii=False))


## Entry Point


In [ ]:
if __name__ == "__main__":
    main()
